# Fine-tune MiniLM Bi-Encoder tren Kaggle

Notebook nay chay doc lap, khong can clone repository. No se:

1. Tai `phdquang/allnli-pair-class-processed` bang `load_dataset()`.
2. Lay cac dong `entailment` lam cap `anchor-positive`.
3. Danh gia TF-IDF va pretrained MiniLM.
4. Fine-tune MiniLM bang `MultipleNegativesRankingLoss`.
5. Danh gia lai va tao bang so sanh.
6. Nen model va ket qua thanh file ZIP.

Truoc khi chay, trong Kaggle hay chon **Settings > Accelerator > GPU** va bat **Internet**.

## 1. Cai thu vien

In [1]:
%pip install -q -U "datasets>=3.0" "sentence-transformers>=5.0,<6" "accelerate>=1.0" "scikit-learn>=1.3" "pandas>=2.0" "pyarrow>=15.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 

## 2. Import va cau hinh

In [2]:
import json
import os
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.sentence_transformer.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers.sentence_transformer.training_args import BatchSamplers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

DATASET_ID = "phdquang/allnli-pair-class-processed"
BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
OUTPUT_ROOT = Path("/kaggle/working/allnli-minilm-biencoder")
FINAL_MODEL_DIR = OUTPUT_ROOT / "final"
RESULTS_DIR = OUTPUT_ROOT / "results"

SEED = 42
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 128
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
MAX_SEQ_LENGTH = 128
EVAL_STEPS = 25
SAVE_STEPS = 25
MAX_RETRIEVAL_QUERIES = 1000
RETRIEVAL_POOL_SIZE = 20

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("GPU chua duoc bat. Chon Kaggle Settings > Accelerator > GPU roi restart session.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("Output:", OUTPUT_ROOT)

GPU: Tesla T4
CUDA: 12.8
Output: /kaggle/working/allnli-minilm-biencoder


## 3. Tai dataset tu Hugging Face

In [3]:
from datasets import load_dataset

ds = load_dataset("phdquang/allnli-pair-class-processed")

train_ds = ds["train"]
dev_ds = ds["dev"]
test_ds = ds["test"]

print(ds)
print("Columns:", train_ds.column_names)
print("First row:", train_ds[0])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/960k [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/880k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/906k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['dataset_name', 'subset', 'split', 'premise', 'hypothesis', 'label', 'premise_clean', 'hypothesis_clean', 'label_name', 'premise_char_len', 'hypothesis_char_len', 'premise_token_len', 'hypothesis_token_len', 'lexical_overlap'],
        num_rows: 5000
    })
    dev: Dataset({
        features: ['dataset_name', 'subset', 'split', 'premise', 'hypothesis', 'label', 'premise_clean', 'hypothesis_clean', 'label_name', 'premise_char_len', 'hypothesis_char_len', 'premise_token_len', 'hypothesis_token_len', 'lexical_overlap'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['dataset_name', 'subset', 'split', 'premise', 'hypothesis', 'label', 'premise_clean', 'hypothesis_clean', 'label_name', 'premise_char_len', 'hypothesis_char_len', 'premise_token_len', 'hypothesis_token_len', 'lexical_overlap'],
        num_rows: 5000
    })
})
Columns: ['dataset_name', 'subset', 'split', 'premise', 'hypothesis', 'label', 'premise_clea

## 4. Chuan bi du lieu train va evaluation

`MultipleNegativesRankingLoss` can cac cap cau cung nghia. Vi dataset la `pair-class`, notebook chi lay nhan `entailment` de tao `anchor-positive`. Dev/test van giu ca ba nhan.

In [4]:
TEXT_A = "premise_clean" if "premise_clean" in train_ds.column_names else "premise"
TEXT_B = "hypothesis_clean" if "hypothesis_clean" in train_ds.column_names else "hypothesis"

def is_entailment(row):
    if "label_name" in row:
        return row["label_name"] == "entailment"
    return int(row["label"]) == 0

train_positive = train_ds.filter(is_entailment)
train_pairs = Dataset.from_dict({
    "anchor": [str(text) for text in train_positive[TEXT_A]],
    "positive": [str(text) for text in train_positive[TEXT_B]],
})

dev_df = dev_ds.to_pandas()
test_df = test_ds.to_pandas()

if "label_name" not in dev_df.columns:
    label_map = {0: "entailment", 1: "neutral", 2: "contradiction"}
    dev_df["label_name"] = dev_df["label"].map(label_map)
    test_df["label_name"] = test_df["label"].map(label_map)

print(f"Train rows ban dau: {len(train_ds):,}")
print(f"Entailment pairs dung de train: {len(train_pairs):,}")
print(f"Dev rows: {len(dev_df):,}")
print(f"Test rows: {len(test_df):,}")
print(pd.Series(train_ds["label_name"]).value_counts())

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Train rows ban dau: 5,000
Entailment pairs dung de train: 1,685
Dev rows: 5,000
Test rows: 5,000
entailment       1685
neutral          1678
contradiction    1637
Name: count, dtype: int64


## 5. Ham danh gia dung chung

In [5]:
POSITIVE_LABEL = "entailment"
NEGATIVE_RETRIEVAL_LABEL = "contradiction"

def choose_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    index = int(np.argmax(f1))
    return float(thresholds[index]), float(f1[index])

def classification_metrics(y_true, scores, threshold):
    predictions = scores >= threshold
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, predictions, average="binary", zero_division=0
    )
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, predictions)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "average_precision": float(average_precision_score(y_true, scores)),
    }

def sentence_pair_scores(model, frame, batch_size=128):
    left = model.encode(
        frame[TEXT_A].fillna("").astype(str).tolist(),
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )
    right = model.encode(
        frame[TEXT_B].fillna("").astype(str).tolist(),
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )
    return np.sum(left * right, axis=1)

def transformer_retrieval_metrics(model, frame, seed, batch_size=128):
    positives = frame[frame["label_name"] == POSITIVE_LABEL].reset_index(drop=True)
    negatives = frame[frame["label_name"] == NEGATIVE_RETRIEVAL_LABEL].reset_index(drop=True)
    rng = np.random.default_rng(seed)
    if len(positives) > MAX_RETRIEVAL_QUERIES:
        positives = positives.iloc[
            rng.choice(len(positives), MAX_RETRIEVAL_QUERIES, replace=False)
        ].reset_index(drop=True)

    query_embeddings = model.encode(
        positives[TEXT_B].astype(str).tolist(), batch_size=batch_size,
        normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True
    )
    positive_embeddings = model.encode(
        positives[TEXT_A].astype(str).tolist(), batch_size=batch_size,
        normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True
    )
    negative_embeddings = model.encode(
        negatives[TEXT_A].astype(str).tolist(), batch_size=batch_size,
        normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True
    )

    ranks = []
    for index, query_embedding in enumerate(query_embeddings):
        distractors = rng.choice(len(negative_embeddings), RETRIEVAL_POOL_SIZE - 1, replace=False)
        scores = np.concatenate((
            [float(positive_embeddings[index] @ query_embedding)],
            negative_embeddings[distractors] @ query_embedding,
        ))
        permutation = rng.permutation(RETRIEVAL_POOL_SIZE)
        relevant_position = int(np.flatnonzero(permutation == 0)[0])
        ranking = np.argsort(-scores[permutation], kind="stable")
        ranks.append(int(np.flatnonzero(ranking == relevant_position)[0]) + 1)

    ranks = np.asarray(ranks)
    return {
        "queries": int(len(ranks)),
        "precision_at_1": float(np.mean(ranks <= 1)),
        "recall_at_5": float(np.mean(ranks <= 5)),
        "mrr": float(np.mean(1.0 / ranks)),
        "mean_rank": float(np.mean(ranks)),
    }

def evaluate_transformer(name, model):
    dev_scores = sentence_pair_scores(model, dev_df, EVAL_BATCH_SIZE)
    test_scores = sentence_pair_scores(model, test_df, EVAL_BATCH_SIZE)
    dev_targets = (dev_df["label_name"] == POSITIVE_LABEL).to_numpy()
    test_targets = (test_df["label_name"] == POSITIVE_LABEL).to_numpy()
    threshold, _ = choose_threshold(dev_targets, dev_scores)
    result = {
        "model": name,
        **classification_metrics(test_targets, test_scores, threshold),
        **transformer_retrieval_metrics(model, test_df, SEED + 1, EVAL_BATCH_SIZE),
    }
    return result, dev_scores, test_scores

## 6. Chay TF-IDF baseline

In [6]:
train_df = train_ds.to_pandas()
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    sublinear_tf=True,
    norm="l2",
)
tfidf.fit(pd.concat([train_df[TEXT_A], train_df[TEXT_B]], ignore_index=True).fillna(""))

def tfidf_pair_scores(frame):
    left = tfidf.transform(frame[TEXT_A].fillna(""))
    right = tfidf.transform(frame[TEXT_B].fillna(""))
    return np.asarray(left.multiply(right).sum(axis=1)).ravel()

def tfidf_retrieval_metrics(frame, seed):
    positives = frame[frame["label_name"] == POSITIVE_LABEL].reset_index(drop=True)
    negatives = frame[frame["label_name"] == NEGATIVE_RETRIEVAL_LABEL].reset_index(drop=True)
    rng = np.random.default_rng(seed)
    if len(positives) > MAX_RETRIEVAL_QUERIES:
        positives = positives.iloc[
            rng.choice(len(positives), MAX_RETRIEVAL_QUERIES, replace=False)
        ].reset_index(drop=True)
    queries = tfidf.transform(positives[TEXT_B].fillna(""))
    relevant = tfidf.transform(positives[TEXT_A].fillna(""))
    distractor_matrix = tfidf.transform(negatives[TEXT_A].fillna(""))
    ranks = []
    for index in range(len(positives)):
        distractors = rng.choice(len(negatives), RETRIEVAL_POOL_SIZE - 1, replace=False)
        scores = np.concatenate((
            [float(relevant[index].dot(queries[index].T).toarray()[0, 0])],
            distractor_matrix[distractors].dot(queries[index].T).toarray().ravel(),
        ))
        permutation = rng.permutation(RETRIEVAL_POOL_SIZE)
        relevant_position = int(np.flatnonzero(permutation == 0)[0])
        ranking = np.argsort(-scores[permutation], kind="stable")
        ranks.append(int(np.flatnonzero(ranking == relevant_position)[0]) + 1)
    ranks = np.asarray(ranks)
    return {
        "queries": int(len(ranks)),
        "precision_at_1": float(np.mean(ranks <= 1)),
        "recall_at_5": float(np.mean(ranks <= 5)),
        "mrr": float(np.mean(1.0 / ranks)),
        "mean_rank": float(np.mean(ranks)),
    }

tfidf_dev_scores = tfidf_pair_scores(dev_df)
tfidf_test_scores = tfidf_pair_scores(test_df)
dev_targets = (dev_df["label_name"] == POSITIVE_LABEL).to_numpy()
test_targets = (test_df["label_name"] == POSITIVE_LABEL).to_numpy()
tfidf_threshold, _ = choose_threshold(dev_targets, tfidf_dev_scores)
tfidf_result = {
    "model": "TF-IDF",
    **classification_metrics(test_targets, tfidf_test_scores, tfidf_threshold),
    **tfidf_retrieval_metrics(test_df, SEED + 1),
}
pd.DataFrame([tfidf_result])

,model,threshold,accuracy,precision,recall,f1,roc_auc,average_precision,queries,precision_at_1,recall_at_5,mrr,mean_rank
0,TF-IDF,0.129885,0.553,0.427704,0.800455,0.557513,0.666524,0.504354,1000,0.842,0.907,0.876565,2.188


## 7. Danh gia pretrained MiniLM

In [7]:
model = SentenceTransformer(BASE_MODEL, device="cuda")
model.max_seq_length = MAX_SEQ_LENGTH

pretrained_result, pretrained_dev_scores, pretrained_test_scores = evaluate_transformer(
    "Pretrained MiniLM", model
)
pd.DataFrame([tfidf_result, pretrained_result])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

,model,threshold,accuracy,precision,recall,f1,roc_auc,average_precision,queries,precision_at_1,recall_at_5,mrr,mean_rank
0,TF-IDF,0.129885,0.5530,0.427704,0.800455,0.557513,0.666524,0.504354,1000,0.842,0.907,0.876565,2.188
1,Pretrained MiniLM,0.579734,0.6774,0.527568,0.794201,0.633991,0.780868,0.642787,1000,0.975,0.997,0.984458,1.054


## 8. Fine-tune Bi-Encoder

In [8]:
score_map = {"entailment": 1.0, "neutral": 0.5, "contradiction": 0.0}
dev_similarity_scores = [score_map[label] for label in dev_df["label_name"]]

evaluator = EmbeddingSimilarityEvaluator(
    sentences1=dev_df[TEXT_A].astype(str).tolist(),
    sentences2=dev_df[TEXT_B].astype(str).tolist(),
    scores=dev_similarity_scores,
    batch_size=EVAL_BATCH_SIZE,
    main_similarity="cosine",
    name="allnli-pair-class-dev",
    show_progress_bar=True,
)

bf16_supported = bool(
    hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported()
)

training_args = SentenceTransformerTrainingArguments(
    output_dir=str(OUTPUT_ROOT / "checkpoints"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    fp16=not bf16_supported,
    bf16=bf16_supported,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=10,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

loss = MultipleNegativesRankingLoss(model)
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_pairs,
    loss=loss,
    evaluator=evaluator,
)

trainer.train()

The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.
Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Allnli-pair-class-dev Pearson Cosine,Allnli-pair-class-dev Spearman Cosine
25,0.238227,No log,0.440483,0.429791


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=42, training_loss=0.21045356563159398, metrics={'train_runtime': 13.7905, 'train_samples_per_second': 366.558, 'train_steps_per_second': 3.046, 'total_flos': 0.0, 'train_loss': 0.21045356563159398, 'epoch': 3.0})

## 9. Luu va danh gia model fine-tuned

In [9]:
model.save_pretrained(str(FINAL_MODEL_DIR))
final_evaluator_metrics = evaluator(model, output_path=str(RESULTS_DIR))

finetuned_result, finetuned_dev_scores, finetuned_test_scores = evaluate_transformer(
    "Fine-tuned MiniLM", model
)

comparison = pd.DataFrame([
    tfidf_result,
    pretrained_result,
    finetuned_result,
])

ordered_columns = [
    "model", "accuracy", "precision", "recall", "f1", "roc_auc",
    "average_precision", "precision_at_1", "recall_at_5", "mrr", "mean_rank",
]
comparison = comparison[ordered_columns]
comparison.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)

test_predictions = test_df[["premise", "hypothesis", "label_name"]].copy()
test_predictions["tfidf_cosine"] = tfidf_test_scores
test_predictions["pretrained_minilm_cosine"] = pretrained_test_scores
test_predictions["finetuned_minilm_cosine"] = finetuned_test_scores
test_predictions.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)

metadata = {
    "dataset_id": DATASET_ID,
    "base_model": BASE_MODEL,
    "train_rows_total": len(train_ds),
    "train_entailment_pairs": len(train_pairs),
    "dev_rows": len(dev_ds),
    "test_rows": len(test_ds),
    "epochs": NUM_EPOCHS,
    "batch_size": TRAIN_BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "max_seq_length": MAX_SEQ_LENGTH,
    "gpu": torch.cuda.get_device_name(0),
    "final_evaluator_metrics": {
        str(key): float(value) if hasattr(value, "item") else value
        for key, value in final_evaluator_metrics.items()
    },
}
(RESULTS_DIR / "training_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

comparison

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

,model,accuracy,precision,recall,f1,roc_auc,average_precision,precision_at_1,recall_at_5,mrr,mean_rank
0,TF-IDF,0.5530,0.427704,0.800455,0.557513,0.666524,0.504354,0.842,0.907,0.876565,2.188
1,Pretrained MiniLM,0.6774,0.527568,0.794201,0.633991,0.780868,0.642787,0.975,0.997,0.984458,1.054
2,Fine-tuned MiniLM,0.6500,0.501541,0.832860,0.626068,0.773784,0.634833,0.973,0.998,0.983583,1.056


## 10. Nen model va ket qua

Sau khi cell nay chay xong, chon **Save Version** tren Kaggle. Hai file ZIP se xuat hien trong Output.

In [10]:
model_zip = shutil.make_archive(
    "/kaggle/working/allnli-minilm-biencoder-model",
    "zip",
    FINAL_MODEL_DIR,
)
results_zip = shutil.make_archive(
    "/kaggle/working/allnli-minilm-biencoder-results",
    "zip",
    RESULTS_DIR,
)

print("Model ZIP:", model_zip)
print("Results ZIP:", results_zip)

Model ZIP: /kaggle/working/allnli-minilm-biencoder-model.zip
Results ZIP: /kaggle/working/allnli-minilm-biencoder-results.zip


## 11. Tuy chon: push model len Hugging Face

Tao Kaggle Secret ten `HF_TOKEN`, doi `PUSH_TO_HUB = True`, va sua `HUB_MODEL_ID`. Khong ghi token truc tiep vao notebook.

In [11]:
PUSH_TO_HUB = False
HUB_MODEL_ID = "phdquang/allnli-minilm-biencoder"
HUB_PRIVATE = True

if PUSH_TO_HUB:
    from kaggle_secrets import UserSecretsClient

    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    model.push_to_hub(
        HUB_MODEL_ID,
        token=hf_token,
        private=HUB_PRIVATE,
        exist_ok=True,
    )
    print("Pushed to:", f"https://huggingface.co/{HUB_MODEL_ID}")
else:
    print("Bo qua push len Hugging Face.")

Bo qua push len Hugging Face.
